In [1]:
import pandas as pd
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

## All Processed (Subset) Data Tables

In [2]:
DATA_PATH = PROJECT_ROOT / 'data'
PROCESSED = DATA_PATH / 'processed'
FINAL = DATA_PATH / 'modeling_datasets' # model ready datasets that will be produced from this step will be stored in here.

afib_meta = pd.read_parquet(PROCESSED / 'afib_subset_metadata.parquet')
norm_meta = pd.read_parquet(PROCESSED / 'norm_subset_metadata.parquet')

afib_feat = pd.read_parquet(PROCESSED / 'afib_subset_features.parquet')
norm_feat = pd.read_parquet(PROCESSED / 'norm_subset_features.parquet')

afib_emb = pd.read_parquet(PROCESSED / 'embeddings' / 'afib_subset_embeddings_exp.parquet')
norm_emb = pd.read_parquet(PROCESSED / 'embeddings' / 'norm_subset_embeddings_exp.parquet')

In [3]:
print(afib_meta.shape)
print(norm_meta.shape)

print(afib_feat.shape)
print(norm_feat.shape)

print(afib_emb.shape)
print(norm_emb.shape)

(10000, 11)
(10000, 11)
(9991, 67)
(9988, 67)
(10000, 515)
(10000, 515)


We dropped some invalid ECG recordings that was found during the signal feature processing so that explains why the feature tables have less rows. We'll ensure to remove those rows are removed from all analysis for fair comparison between signal processing features vs representation learning. The embeddings are that resulted from those ECGs are also likely bad.

In [4]:
afib_set = set(afib_feat['waveform_path'])

afib_meta = afib_meta[afib_meta['waveform_path'].isin(afib_set)]
afib_emb = afib_emb[afib_emb['waveform_path'].isin(afib_set)]


norm_set = set(norm_feat['waveform_path'])

norm_meta = norm_meta[norm_meta['waveform_path'].isin(norm_set)]
norm_emb = norm_emb[norm_emb['waveform_path'].isin(norm_set)]

In [5]:
print(afib_meta.shape)
print(norm_meta.shape)

print(afib_feat.shape)
print(norm_feat.shape)

print(afib_emb.shape)
print(norm_emb.shape)

(9991, 11)
(9988, 11)
(9991, 67)
(9988, 67)
(9991, 515)
(9988, 515)


In [6]:
assert all(afib_meta.columns == norm_meta.columns)
assert all(afib_feat.columns == norm_feat.columns)
assert all(afib_emb.columns == norm_emb.columns)

print("META COLUMNS")
print(afib_meta.columns.tolist())

print("\nFEATURE COLUMNS")
print(afib_feat.columns.tolist())

print("\nEMB COLUMNS")
print(afib_emb.columns.tolist())

META COLUMNS
['subject_id', 'study_id', 'file_name', 'ecg_time', 'path', 'machine_report', 'is_af', 'is_normal_strict', 'is_clearly_abnormal', 'waveform_path', 'label']

FEATURE COLUMNS
['subject_id', 'study_id', 'path', 'waveform_path', 'label', 'lead_mean_mean', 'lead_mean_std', 'lead_mean_min', 'lead_mean_max', 'lead_std_mean', 'lead_std_std', 'lead_std_min', 'lead_std_max', 'lead_min_mean', 'lead_min_std', 'lead_min_min', 'lead_min_max', 'lead_max_mean', 'lead_max_std', 'lead_max_min', 'lead_max_max', 'lead_ptp_mean', 'lead_ptp_std', 'lead_ptp_min', 'lead_ptp_max', 'lead_rms_mean', 'lead_rms_std', 'lead_rms_min', 'lead_rms_max', 'lead_energy_mean', 'lead_energy_std', 'lead_energy_min', 'lead_energy_max', 'dom_freq_mean', 'dom_freq_std', 'dom_freq_min', 'dom_freq_max', 'spec_entropy_mean', 'spec_entropy_std', 'spec_entropy_min', 'spec_entropy_max', 'spec_centroid_mean', 'spec_centroid_std', 'spec_centroid_min', 'spec_centroid_max', 'bp_0_5_5_mean', 'bp_0_5_5_std', 'bp_0_5_5_min', 'b

'waveform_path' will be our unique identifier as both tables were generated by loading ecg through it. 

### Merge Metadata Into Signal Feature & Embedding Tables

In [7]:
afib_signal = afib_meta.merge(afib_feat, on='waveform_path', how='inner', suffixes=("", "_dup"))
afib_signal.columns

Index(['subject_id', 'study_id', 'file_name', 'ecg_time', 'path',
       'machine_report', 'is_af', 'is_normal_strict', 'is_clearly_abnormal',
       'waveform_path', 'label', 'subject_id_dup', 'study_id_dup', 'path_dup',
       'label_dup', 'lead_mean_mean', 'lead_mean_std', 'lead_mean_min',
       'lead_mean_max', 'lead_std_mean', 'lead_std_std', 'lead_std_min',
       'lead_std_max', 'lead_min_mean', 'lead_min_std', 'lead_min_min',
       'lead_min_max', 'lead_max_mean', 'lead_max_std', 'lead_max_min',
       'lead_max_max', 'lead_ptp_mean', 'lead_ptp_std', 'lead_ptp_min',
       'lead_ptp_max', 'lead_rms_mean', 'lead_rms_std', 'lead_rms_min',
       'lead_rms_max', 'lead_energy_mean', 'lead_energy_std',
       'lead_energy_min', 'lead_energy_max', 'dom_freq_mean', 'dom_freq_std',
       'dom_freq_min', 'dom_freq_max', 'spec_entropy_mean', 'spec_entropy_std',
       'spec_entropy_min', 'spec_entropy_max', 'spec_centroid_mean',
       'spec_centroid_std', 'spec_centroid_min', 'spec_c

In [8]:
afib_hubert = afib_meta.merge(afib_emb, on='waveform_path', how='inner', suffixes=("", "_dup"))
print(afib_hubert.columns.to_list())

['subject_id', 'study_id', 'file_name', 'ecg_time', 'path', 'machine_report', 'is_af', 'is_normal_strict', 'is_clearly_abnormal', 'waveform_path', 'label', 'file_name_dup', 'embedding', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'emb_19', 'emb_20', 'emb_21', 'emb_22', 'emb_23', 'emb_24', 'emb_25', 'emb_26', 'emb_27', 'emb_28', 'emb_29', 'emb_30', 'emb_31', 'emb_32', 'emb_33', 'emb_34', 'emb_35', 'emb_36', 'emb_37', 'emb_38', 'emb_39', 'emb_40', 'emb_41', 'emb_42', 'emb_43', 'emb_44', 'emb_45', 'emb_46', 'emb_47', 'emb_48', 'emb_49', 'emb_50', 'emb_51', 'emb_52', 'emb_53', 'emb_54', 'emb_55', 'emb_56', 'emb_57', 'emb_58', 'emb_59', 'emb_60', 'emb_61', 'emb_62', 'emb_63', 'emb_64', 'emb_65', 'emb_66', 'emb_67', 'emb_68', 'emb_69', 'emb_70', 'emb_71', 'emb_72', 'emb_73', 'emb_74', 'emb_75', 'emb_76', 'emb_77', 'emb_78', 'emb_79', 'emb_80', 'emb_81', 'emb

In [9]:
afib_signal = afib_signal.drop(columns=['subject_id_dup', 'study_id_dup', 'path_dup', 'label_dup']).copy()
afib_hubert = afib_hubert.drop(columns=['file_name_dup', 'embedding']).copy()

In [10]:
norm_signal = norm_meta.merge(norm_feat, on='waveform_path', how='inner', suffixes=("", "_dup"))
norm_hubert = norm_meta.merge(norm_emb, on='waveform_path', how='inner', suffixes=("", "_dup"))

norm_signal = norm_signal.drop(columns=['subject_id_dup', 'study_id_dup', 'path_dup', 'label_dup']).copy()
norm_hubert = norm_hubert.drop(columns=['file_name_dup', 'embedding']).copy()

## Current Datasets
#### Afib Detection Task
- **afib_signal**
- **afib_hubert**
#### Abnormal ECG Task
- **norm_signal**
- **norm_hubert**

With these four datasets, we can compare the performance of signal processing features vs representation learning features againt the two tasks. We'll also have a third dataset with both signal and representation features for additional insight on performance. 

In [11]:
afib_combined = afib_signal.merge(afib_emb.drop(columns=['file_name']), on='waveform_path', how='inner', suffixes=('', '_dup'))
print(afib_combined.shape)

(9991, 586)


In [12]:
norm_combined = norm_signal.merge(norm_emb.drop(columns=['file_name']), on='waveform_path', how='inner', suffixes=('', '_dup'))
print(norm_combined.shape)

(9988, 586)


## Save Final Model Datasets
**Important:** Some columns must be dropped before being put into any models to prevent label leakage and potentially learning relationships between subjects and conditions. Here's a list of all columns that must be dropped before fitting a model.

**Identifiers**
- subject_id
- study_id
- file_name
- path
- waveform_path

**Labels**
- is_af
- is_normal_strict
- is_clearly_abnormal
- machine_report
- label

**Also**
- ecg_time, because the model may learn patterns in time and ecg results

In [13]:
afib_signal.to_parquet(FINAL / 'afib_features_signal.parquet', index=False)
afib_hubert.to_parquet(FINAL / 'afib_features_hubert.parquet', index=False)
afib_combined.to_parquet(FINAL / 'afib_features_combined.parquet', index=False)

norm_signal.to_parquet(FINAL / 'abnorm_features_signal.parquet', index=False)
norm_hubert.to_parquet(FINAL / 'abnorm_features_hubert.parquet', index=False)
norm_combined.to_parquet(FINAL / 'abnorm_features_combined.parquet', index=False)